In [0]:
%sql
USE CATALOG databricksproject;

## Create a schema

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS taxi_db;

In [0]:
%sql
USE SCHEMA taxi_db;

### Reading and Writing  Data to Cloud Storage using delta tables

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import *

In [0]:
silver = 'abfss://silver@nyctaxidatastorage.dfs.core.windows.net/'
gold = 'abfss://gold@nyctaxidatastorage.dfs.core.windows.net/'

In [0]:
df_trip_zone = spark.read.format('parquet')\
                .option('inferSchema',True)\
                .option('header',True)\
                .load(f'{silver}/trip_zone')

In [0]:
display(df_trip_zone)

LocationID,Borough,service_zone,zone1,zone2
1,EWR,EWR,Newark Airport,null
2,Queens,Boro Zone,Jamaica Bay,null
3,Bronx,Boro Zone,Allerton,Pelham Gardens
4,Manhattan,Yellow Zone,Alphabet City,null
5,Staten Island,Boro Zone,Arden Heights,null
6,Staten Island,Boro Zone,Arrochar,Fort Wadsworth
7,Queens,Boro Zone,Astoria,null
8,Queens,Boro Zone,Astoria Park,null
9,Queens,Boro Zone,Auburndale,null
10,Queens,Boro Zone,Baisley Park,null


In [0]:
# write to delta table
df_trip_zone.write.format('delta')\
        .mode('overwrite')\
        .option('path',f'{gold}/trip_zone')\
        .saveAsTable('trip_zone')

LocationID,Borough,service_zone,zone1,zone2
1,EWR,EWR,Newark Airport,null
2,Queens,Boro Zone,Jamaica Bay,null
3,Bronx,Boro Zone,Allerton,Pelham Gardens
4,Manhattan,Yellow Zone,Alphabet City,null
5,Staten Island,Boro Zone,Arden Heights,null
6,Staten Island,Boro Zone,Arrochar,Fort Wadsworth
7,Queens,Boro Zone,Astoria,null
8,Queens,Boro Zone,Astoria Park,null
9,Queens,Boro Zone,Auburndale,null
10,Queens,Boro Zone,Baisley Park,null


**Trip Type**

In [0]:
df_trip_type = spark.read.format('parquet')\
                .option('inferSchema',True)\
                .option('header',True)\
                .load(f'{silver}/trip_type')

In [0]:
display(df_trip_type)


trip_type,trip_description
1,Street-hail
2,Dispatch


In [0]:
df_trip_type.write.format('delta')\
        .mode('append')\
        .option('path',f'{gold}/trip_type')\
        .saveAsTable('trip_type')

**Trips Data**

In [0]:
df_trip = spark.read.format('parquet')\
                .option('inferSchema',True)\
                .option('header',True)\
                .load(f'{silver}/trip_data')

In [0]:
df_trip.write.format('delta')\
        .mode('append')\
        .option('path',f'{gold}/tripsdata')\
        .saveAsTable('trip_data')

In [0]:
%sql
SELECT *
FROM trip_data;

VendorID,PULocationID,DOLocationID,fare_amount,total_amount
2,74,42,7.2,11.64
2,74,42,7.2,9.7
2,83,160,13.5,21.0
2,166,127,24.7,27.7
1,166,262,18.4,24.65
1,112,48,26.8,39.35
2,83,87,43.6,59.52
2,66,233,28.9,41.88
2,223,223,5.1,9.12
2,130,130,7.9,10.4


## Exploring Delta Lake

**Versioning**

In [0]:
%sql
select * from trip_zone 
where LocationID = 10

LocationID,Borough,service_zone,zone1,zone2
10,Queens,Boro Zone,Baisley Park,null


In [0]:
%sql
UPDATE trip_zone 
SET Borough = 'EMR' where LocationID = 10;

num_affected_rows
1


In [0]:
%sql
DELETE FROM trip_zone 
WHERE LocationID = 1

num_affected_rows
1


**Versioning**

In [0]:
spark.sql("DESCRIBE HISTORY trip_zone").display()

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-06-23T04:21:40Z,146154291954793,so2439a@american.edu,DELETE,"Map(predicate -> [""(LocationID#2106 = 1)""])",null,List(4155170020452964),0621-150002-9gw58ruo,1,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 1, numAddedChangeFiles -> 0, executionTimeMs -> 1629, numDeletionVectorsUpdated -> 1, numDeletedRows -> 1, scanTimeMs -> 1258, numAddedFiles -> 0, numAddedBytes -> 0, rewriteTimeMs -> 370)",null,Databricks-Runtime/17.3.x-photon-scala2.13
1,2026-06-23T04:21:11Z,146154291954793,so2439a@american.edu,UPDATE,"Map(predicate -> [""(LocationID#1785 = 10)""])",null,List(4155170020452964),0621-150002-9gw58ruo,0,WriteSerializable,false,"Map(numRemovedFiles -> 0, numRemovedBytes -> 0, numCopiedRows -> 0, numDeletionVectorsAdded -> 1, numDeletionVectorsRemoved -> 0, numAddedChangeFiles -> 0, executionTimeMs -> 6156, numDeletionVectorsUpdated -> 0, scanTimeMs -> 3448, numAddedFiles -> 1, numUpdatedRows -> 1, numAddedBytes -> 1632, rewriteTimeMs -> 2678)",null,Databricks-Runtime/17.3.x-photon-scala2.13
0,2026-06-23T04:06:48Z,146154291954793,so2439a@american.edu,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> false, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4155170020452964),0621-150002-9gw58ruo,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 265, numOutputBytes -> 6491)",null,Databricks-Runtime/17.3.x-photon-scala2.13


In [0]:
%sql
select * from trip_zone
where LocationID = 1

LocationID,Borough,service_zone,zone1,zone2


**Time Travel**

In [0]:
%sql
RESTORE trip_zone TO VERSION AS OF 0

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
6491,1,2,1,8123,6491


In [0]:
%sql
SELECT * from trip_zone

LocationID,Borough,service_zone,zone1,zone2
1,EWR,EWR,Newark Airport,null
2,Queens,Boro Zone,Jamaica Bay,null
3,Bronx,Boro Zone,Allerton,Pelham Gardens
4,Manhattan,Yellow Zone,Alphabet City,null
5,Staten Island,Boro Zone,Arden Heights,null
6,Staten Island,Boro Zone,Arrochar,Fort Wadsworth
7,Queens,Boro Zone,Astoria,null
8,Queens,Boro Zone,Astoria Park,null
9,Queens,Boro Zone,Auburndale,null
10,Queens,Boro Zone,Baisley Park,null


## Delta Tables

In [0]:
%sql
SELECT *
FROM trip_zone

LocationID,Borough,service_zone,zone1,zone2
1,EWR,EWR,Newark Airport,null
2,Queens,Boro Zone,Jamaica Bay,null
3,Bronx,Boro Zone,Allerton,Pelham Gardens
4,Manhattan,Yellow Zone,Alphabet City,null
5,Staten Island,Boro Zone,Arden Heights,null
6,Staten Island,Boro Zone,Arrochar,Fort Wadsworth
7,Queens,Boro Zone,Astoria,null
8,Queens,Boro Zone,Astoria Park,null
9,Queens,Boro Zone,Auburndale,null
10,Queens,Boro Zone,Baisley Park,null


In [0]:
%sql
SELECT *
FROM trip_type

trip_type,trip_description
1,Street-hail
2,Dispatch


In [0]:
%sql
SELECT *
FROM trip_data

VendorID,PULocationID,DOLocationID,fare_amount,total_amount
2,74,42,7.2,11.64
2,74,42,7.2,9.7
2,83,160,13.5,21.0
2,166,127,24.7,27.7
1,166,262,18.4,24.65
1,112,48,26.8,39.35
2,83,87,43.6,59.52
2,66,233,28.9,41.88
2,223,223,5.1,9.12
2,130,130,7.9,10.4
